<a href="https://colab.research.google.com/github/Ayesha-Anwar607/Ayesha-Anwar607/blob/main/Data_Cleaning_and_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Recommender file/final_movie.csv', encoding='latin1')
df.head(5)


,movieId_x,title,genres,movieId_y,release_date,overview,popularity,vote_average,vote_count,tmdb_id,cast,director
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0,11/22/1995,"Led by Woody, Andy's toys live happily in his ...",29.4626,7.970,19082.0,862.0,"Tom Hanks, Tim Allen, Don Rickles",John Lasseter
1,2,Jumanji (1995),Adventure|Children|Fantasy,2.0,12/15/1995,When siblings Judy and Peter discover an encha...,3.2037,7.240,10881.0,8844.0,"Robin Williams, Kirsten Dunst, Bradley Pierce",Joe Johnston
2,3,Grumpier Old Men (1995),Comedy|Romance,3.0,12/22/1995,A family wedding reignites the ancient feud be...,2.4286,6.467,403.0,15602.0,"Walter Matthau, Jack Lemmon, Ann-Margret",Howard Deutch
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,4.0,12/22/1995,"Cheated on, mistreated and stepped on, the wom...",1.4808,6.267,174.0,31357.0,"Whitney Houston, Angela Bassett, Loretta Devine",Forest Whitaker
4,5,Father of the Bride Part II (1995),Comedy,5.0,12/8/1995,Just when George Banks has recovered from his ...,1.6454,6.252,763.0,11862.0,"Steve Martin, Diane Keaton, Martin Short",Charles Shyer


In [ ]:
df.shape # Main combined dataset with cast,director


(2215, 12)

### Importing Libraries

In [ ]:
import os, re, json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from scipy import sparse


In [ ]:
# NLTK stuff
import nltk

# Downloads (will be quick; needed for tokenization/lemmatization/stopwords)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# make folders to save artifacts
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [ ]:
# Cell 2: load your final merged CSV and inspect
print("Rows,Cols:", df.shape)
print("Columns:", df.columns.tolist())
df.head().T  # show first row transposed for quick glance

Rows,Cols: (2215, 12)
Columns: ['movieId_x', 'title', 'genres', 'movieId_y', 'release_date', 'overview', 'popularity', 'vote_average', 'vote_count', 'tmdb_id', 'cast', 'director']


,0,1,2,3,4
movieId_x,1,2,3,4,5
title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995)
genres,Adventure|Animation|Children|Comedy|Fantasy,Adventure|Children|Fantasy,Comedy|Romance,Comedy|Drama|Romance,Comedy
movieId_y,1.0,2.0,3.0,4.0,5.0
release_date,11/22/1995,12/15/1995,12/22/1995,12/22/1995,12/8/1995
overview,"Led by Woody, Andy's toys live happily in his ...",When siblings Judy and Peter discover an encha...,A family wedding reignites the ancient feud be...,"Cheated on, mistreated and stepped on, the wom...",Just when George Banks has recovered from his ...
popularity,29.4626,3.2037,2.4286,1.4808,1.6454
vote_average,7.97,7.24,6.467,6.267,6.252
vote_count,19082.0,10881.0,403.0,174.0,763.0
tmdb_id,862.0,8844.0,15602.0,31357.0,11862.0


In [ ]:
# Cell 3: auto-detect relevant columns (safe mapping)
def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

title_col = first_existing_col(df, ["title","name","movie_title","original_title"])
overview_col = first_existing_col(df, ["overview","description","plot","summary"])
genres_col = first_existing_col(df, ["genres","genre","genres_x","movie_genres","genres_list"])
cast_col = first_existing_col(df, ["cast","casts","actors","top_cast","cast_list"])
director_col = first_existing_col(df, ["director","directors","crew_director","Director","directed_by"])

print("Mapped columns:")
print(" title:", title_col)
print(" overview:", overview_col)
print(" genres:", genres_col)
print(" cast:", cast_col)
print(" director:", director_col)


Mapped columns:
 title: title
 overview: overview
 genres: genres
 cast: cast
 director: director


In [ ]:
# Cell 4: parsing helpers + text cleaner
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def parse_genres(val):
    if pd.isna(val): return ""
    if isinstance(val, list):
        return " ".join([str(x).replace(" ", "_") for x in val])
    s = str(val).strip()
    # if JSON-like list of dicts from TMDb
    if s.startswith("[") or s.startswith("{"):
        try:
            j = json.loads(s)
            # TMDb style: [{"id": 18, "name": "Drama"}, ...]
            if isinstance(j, list):
                names = [it.get("name") if isinstance(it, dict) else str(it) for it in j]
                names = [n for n in names if n]
                return " ".join([n.replace(" ", "_") for n in names])
            if isinstance(j, dict) and "genres" in j:
                return " ".join([g.get("name","").replace(" ", "_") for g in j["genres"]])
        except Exception:
            pass
    # pipe or comma separated
    if "|" in s:
        parts = [p.strip().replace(" ", "_") for p in s.split("|") if p.strip()]
        return " ".join(parts)
    if "," in s:
        parts = [p.strip().replace(" ", "_") for p in s.split(",") if p.strip()]
        return " ".join(parts)
    return s.replace(" ", "_")

def parse_cast(val, top_k=3):
    if pd.isna(val): return ""
    if isinstance(val, list):
        return " ".join([str(x).replace(" ", "_") for x in val[:top_k]])
    s = str(val).strip()
    if s.startswith("["):
        try:
            j = json.loads(s)
            if isinstance(j, list):
                names = []
                for it in j[:top_k]:
                    if isinstance(it, dict):
                        n = it.get("name") or it.get("original_name") or it.get("actor")
                        if n: names.append(n.replace(" ", "_"))
                    else:
                        names.append(str(it).replace(" ", "_"))
                return " ".join(names)
        except Exception:
            pass
    # comma/pipe separated simple cases
    if "," in s:
        parts = [p.strip().replace(" ", "_") for p in s.split(",")[:top_k] if p.strip()]
        return " ".join(parts)
    if "|" in s:
        parts = [p.strip().replace(" ", "_") for p in s.split("|")[:top_k] if p.strip()]
        return " ".join(parts)
    # fallback: return first 3 words joined by underscore
    return "_".join(s.split()[:top_k])

def parse_director(val):
    if pd.isna(val): return ""
    if isinstance(val, list):
        return str(val[0]).replace(" ", "_")
    s = str(val).strip()
    if s.startswith("["):
        try:
            j = json.loads(s)
            # search for crew-like dicts with job Director
            if isinstance(j, list):
                for it in j:
                    if isinstance(it, dict) and (it.get("job","").lower() == "director" or it.get("department","").lower()=="directing"):
                        n = it.get("name")
                        if n: return n.replace(" ", "_")
                # fallback to first name
                if j and isinstance(j[0], dict) and "name" in j[0]:
                    return j[0]["name"].replace(" ", "_")
        except Exception:
            pass
    if "," in s:
        return s.split(",")[0].strip().replace(" ", "_")
    if "|" in s:
        return s.split("|")[0].strip().replace(" ", "_")
    return s.replace(" ", "_")

def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)            # remove html
    text = re.sub(r"http\S+", " ", text)           # remove urls
    text = re.sub(r"[^a-z0-9_\s]", " ", text)      # keep a-z, digits, underscores and spaces
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)


## Content Based Filtering with overview

In [ ]:
 import nltk
 nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# # Cell 5: build content columns
# content_raw = []
# content_clean = []

# N = len(df)
# print(f"Processing {N} rows...")

# for i, row in df.iterrows():
#     parts = []
#     # title (keep original title as is)
#     if title_col:
#         parts.append(str(row.get(title_col, "")))
#     # genres -> parsed
#     if genres_col:
#         parts.append(parse_genres(row.get(genres_col, "")))
#     # cast -> top-3
#     if cast_col:
#         parts.append(parse_cast(row.get(cast_col, ""), top_k=3))
#     # director
#     if director_col:
#         parts.append(parse_director(row.get(director_col, "")))
#     # overview / plot
#     if overview_col:
#         parts.append(str(row.get(overview_col, "")))

#     # also include common extras if present
#     for extra in ["keywords", "tags", "plot_keywords"]:
#         if extra in df.columns:
#             parts.append(str(row.get(extra, "")))

#     raw = " ".join([p for p in parts if p])
#     clean = clean_text(raw)

#     content_raw.append(raw)
#     content_clean.append(clean)

#     # small progress print every 500 rows
#     if (i+1) % 500 == 0:
#         print(f"Processed {i+1}/{N} movies")

# df["content_raw"] = content_raw
# df["content"] = content_clean

# # Save cleaned DF
# OUT_PATH = "final_movie_cleaned.csv"
# df.to_csv(OUT_PATH, index=False)
# print("Saved cleaned dataframe to:", OUT_PATH)


Processing 2215 rows...
Processed 500/2215 movies
Processed 1000/2215 movies
Processed 1500/2215 movies
Processed 2000/2215 movies
Saved cleaned dataframe to: final_movie_cleaned.csv


In [ ]:
# Cell 6 (optional): TF-IDF on `content` and save artifacts
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X = tfidf.fit_transform(df["content"].fillna(""))

# save vectorizer + sparse matrix
joblib.dump(tfidf, "/content/drive/MyDrive/Recommender file/models/tfidf_vectorizer.joblib")
sparse.save_npz("/content/drive/MyDrive/Recommender file/models/tfidf_matrix.npz", X)

print("TF-IDF vectorizer saved -> models/tfidf_vectorizer.joblib")
print("TF-IDF matrix saved -> models/tfidf_matrix.npz")
print("Matrix shape:", X.shape)


TF-IDF vectorizer saved -> models/tfidf_vectorizer.joblib
TF-IDF matrix saved -> models/tfidf_matrix.npz
Matrix shape: (2215, 20000)


In [ ]:
df.sample(1)[["title","content_raw","content"]].T


,1359
title,As Good as It Gets (1997)
content_raw,As Good as It Gets (1997) Comedy Drama Romance...
content,good get 1997 comedy drama romance jack_nichol...


In [ ]:
import joblib
import scipy.sparse as sp

# Load vectorizer
tfidf = joblib.load("/content/drive/MyDrive/Recommender file/models/tfidf_vectorizer.joblib")

# Load TF-IDF matrix
tfidf_matrix = sp.load_npz("/content/drive/MyDrive/Recommender file/models/tfidf_matrix.npz")

print("Reloaded TF-IDF matrix:", tfidf_matrix.shape)


Reloaded TF-IDF matrix: (2215, 20000)


In [ ]:
# --- Step 4b: Cosine Similarity + Recommendation Function ---

from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix (all movies vs all movies)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a reverse mapping of movie titles to index
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def recommend_movies(title, num_recommendations=10):
    """
    Recommend movies based on content similarity (TF-IDF + Cosine).
    """
    # Get index of the movie
    idx = indices[title]

    # Get similarity scores for this movie with all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score (highest first, skip itself)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]

    # Get the indices of recommended movies
    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices]



In [ ]:
# --- Example test ---
print("Recommendations for 'Pinocchio (1940)':")
print(recommend_movies("All Dogs Go to Heaven 2 (1996)", 10))


Recommendations for 'Pinocchio (1940)':
1632        All Dogs Go to Heaven (1989)
560     James and the Giant Peach (1996)
807           Alice in Wonderland (1951)
521                     Pinocchio (1940)
801       Sword in the Stone, The (1963)
1625       Lord of the Rings, The (1978)
1639    Adventures in Babysitting (1987)
798                    Cinderella (1950)
568                     Space Jam (1996)
1200                       Shiloh (1997)
Name: title, dtype: object


### Content filtering with Cast,Director and Genre


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import scipy.sparse as sp

# --- 1. Combine text features ---
# Ensure these columns exist in df
for col in ['overview', 'cast', 'director', 'genres']:
    df[col] = df[col].fillna('')

# Create a new combined column
df['combined_features'] = (
    df['overview'] + " " +
    df['cast'] + " " +
    df['director'] + " " +
    df['genres']
)

# --- 2. TF-IDF on combined features ---
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print("New TF-IDF shape:", tfidf_matrix.shape)

# --- 3. Save vectorizer + matrix ---
joblib.dump(tfidf, "/content/drive/MyDrive/Recommender file/models/tfidf_vectorizer_combined.joblib")
sp.save_npz("/content/drive/MyDrive/Recommender file/models/tfidf_matrix_combined.npz", tfidf_matrix)

print("✅ TF-IDF for combined features saved.")


New TF-IDF shape: (2215, 10000)
✅ TF-IDF for combined features saved.


In [ ]:
# --- Reload with combined TF-IDF ---
tfidf = joblib.load("/content/drive/MyDrive/Recommender file/models/tfidf_vectorizer_combined.joblib")
tfidf_matrix = sp.load_npz("/content/drive/MyDrive/Recommender file/models/tfidf_matrix_combined.npz")

# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Reverse index
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

from difflib import get_close_matches

In [ ]:


def recommend_movies(title, num_recommendations=10):
    # If exact match not found, use fuzzy matching
    if title not in indices:
        close_matches = get_close_matches(title, df['title'].tolist(), n=1, cutoff=0.6)
        if not close_matches:
            return f"No close match found for '{title}'"
        title = close_matches[0]
        print(f"🔍 Using closest match: {title}")

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]
    movie_indices = [i[0] for i in sim_scores]

    return df[['title', 'cast', 'director', 'genres']].iloc[movie_indices]

# Example test
recommend_movies("Batman", 10)



🔍 Using closest match: Batman (1989)


,title,cast,director,genres
1207,Batman & Robin (1997),"George Clooney, Chris O'Donnell, Arnold Schwar...",Joel Schumacher,Action|Adventure|Fantasy|Thriller
1087,Batman Returns (1992),"Michael Keaton, Danny DeVito, Michelle Pfeiffer",Tim Burton,Action|Crime
129,Batman Forever (1995),"Val Kilmer, Tommy Lee Jones, Jim Carrey",Joel Schumacher,Action|Adventure|Comedy|Crime
1320,Midnight in the Garden of Good and Evil (1997),"John Cusack, Kevin Spacey, Jude Law",Clint Eastwood,Crime|Drama|Mystery
887,Entertaining Angels: The Dorothy Day Story (1996),"Moira Kelly, Heather Graham, Melinda Dillon",Michael Ray Rhodes,Drama
1353,B. Monkey (1998),"Asia Argento, Jared Harris, Rupert Everett",Michael Radford,Crime|Romance|Thriller
1214,Face/Off (1997),"John Travolta, Nicolas Cage, Joan Allen",John Woo,Action|Crime|Drama|Thriller
162,"Prophecy, The (1995)","Ahn Hyo-seop, Lee Min-ho, Chae Soo-bin",Kim Byung-woo,Fantasy|Horror|Mystery
1065,Blood and Wine (Blood & Wine) (1996),"Jack Nicholson, Stephen Dorff, Jennifer Lopez",Bob Rafelson,Crime|Drama|Thriller
1293,Witness (1985),"Harrison Ford, Kelly McGillis, Josef Sommer",Peter Weir,Drama|Romance|Thriller


## **Content based Filtering is working all well and good.**

## Collaborative Filtering

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load MovieLens datasets
ratings = pd.read_csv("/content/drive/MyDrive/Recommender file/ratings_clean.csv")   # userId, movieId, rating
movies = pd.read_csv("/content/drive/MyDrive/Recommender file/movies.csv")     # movieId, title


In [ ]:
# movies= movies.drop('genres', axis=1)
# ratings= ratings.drop('timestamp', axis=1)

In [ ]:

ratings= ratings.sort_values(by="movieId")

# # Remove last 10 rows
# ratings=ratings.iloc[:-7]

# # Save back to CSV
# ratings.to_csv("ratings_clean.csv", index=False)


In [ ]:
print(ratings.shape, movies.shape)

(2129, 3) (2129, 2)


### item-based CF

In [ ]:
# Create item-user matrix
item_user_matrix = ratings.pivot_table(index="movieId", columns="userId", values="rating").fillna(0)

In [ ]:
# Compute cosine similarity (movie × movie)
item_cosine_sim = cosine_similarity(item_user_matrix)
item_sim_df = pd.DataFrame(item_cosine_sim, index=item_user_matrix.index, columns=item_user_matrix.index)

In [ ]:
def recommend_similar_movies(movie_title, k=10):
    """Recommend movies similar to a given movie (item-based CF)."""
    if movie_title not in movies['title'].values:
        return f"Movie '{movie_title}' not found in dataset!"

    movie_id = movies[movies['title'] == movie_title]['movieId'].values[0]
    scores = item_sim_df[movie_id].sort_values(ascending=False)
    top_ids = scores.iloc[1:k+1].index
    return movies[movies['movieId'].isin(top_ids)]

# Example
print("\n🎬 Item-based recommendations for Toy Story:")
print(recommend_similar_movies("Get Shorty (1995)"))


🎬 Item-based recommendations for Toy Story:
    movieId                                      title
0         1                           Toy Story (1995)
5         6                                Heat (1995)
9        10                           GoldenEye (1995)
10       11             American President, The (1995)
16       17               Sense and Sensibility (1995)
31       32  Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
32       34                                Babe (1995)
33       36                    Dead Man Walking (1995)
35       39                            Clueless (1995)
41       45                          To Die For (1995)


**Item based CF is working well**

### User Based CF

In [ ]:
# Create user-item matrix
user_item_matrix = ratings.pivot_table(index="userId", columns="movieId", values="rating").fillna(0)


In [ ]:
# Compute cosine similarity (user × user)
user_cosine_sim = cosine_similarity(user_item_matrix)
user_sim_df = pd.DataFrame(user_cosine_sim, index=user_item_matrix.index, columns=user_item_matrix.index)

In [ ]:
def recommend_for_user(user_id, k=10, top_n=5):
    if user_id not in user_item_matrix.index:
        return f"User {user_id} not found!"

    # Find most similar users (skip self)
    sim_users = user_sim_df[user_id].sort_values(ascending=False).iloc[1:]

    # Pick top N similar users
    top_users = sim_users.head(top_n).index

    # Movies watched by target user
    target_movies = set(ratings[ratings['userId'] == user_id]['movieId'])

    # Movies watched by top similar users
    similar_user_movies = ratings[ratings['userId'].isin(top_users)]

    # Filter unseen movies
    unseen_movies = similar_user_movies[~similar_user_movies['movieId'].isin(target_movies)]

    if unseen_movies.empty:
        return unseen_movies  # empty DataFrame

    # Sort by avg rating across similar users
    recommendations = (
        unseen_movies.groupby("movieId")["rating"].mean()
        .sort_values(ascending=False)
        .head(k)
        .reset_index()
        .merge(movies, on="movieId")
    )

    return recommendations[["title", "rating"]]


In [ ]:
print(recommend_for_user(9, k=10, top_n=5))


                                               title  rating
0                                        Heat (1995)    5.00
1                                      Casino (1995)    4.50
2  City of Lost Children, The (Cité des enfants p...    4.50
3                                  Get Shorty (1995)    4.00
4          Twelve Monkeys (a.k.a. 12 Monkeys) (1995)    4.00
5                                        Babe (1995)    4.00
6                                   Toy Story (1995)    4.00
7                                     Copycat (1995)    3.50
8                                   GoldenEye (1995)    3.25
9                            Dead Man Walking (1995)    3.00


**user-based is for learning purpose only. mainly we will be usingg item-based filtering.**

**so far as we have come. both filters are working well independently.**

## Hybrid Model

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Recommender file/final_movie_cleaned.csv', encoding='latin1')

ratings = pd.read_csv("/content/drive/MyDrive/Recommender file/ratings_clean.csv")


In [ ]:
# ratings= ratings.drop('timestamp', axis=1)


In [ ]:

# ratings= ratings.sort_values(by="movieId")

# # Remove last 10 rows
# ratings=ratings.iloc[:-1]

# # Save back to CSV
# ratings.to_csv("ratings_clean.csv", index=False)


In [ ]:

print(ratings.shape,df.shape)

(2129, 3) (2215, 14)


In [ ]:
def hybrid_recommend_fixed(user_id, movie_title, k=10, alpha=0.5, exclude_already_rated=True):
    """
    Balanced hybrid recommender:
    - alpha: weight for content (0..1), 1=only content, 0=only collaborative
    - rank-based normalization for fair blending
    """
    # 1) Resolve seed title
    best_title, seed_idx = find_title_best(movie_title)
    if seed_idx is None:
        return f"Movie '{movie_title}' not found."
    seed_mid = df.iloc[seed_idx][movie_id_col]

    # 2) Get raw scores (force dicts)
    content_raw = get_content_scores(best_title)
    collab_raw  = get_collab_predicted_scores(user_id)

    if isinstance(content_raw, pd.Series):
        content_raw = content_raw.to_dict()
    if isinstance(collab_raw, pd.Series):
        collab_raw = collab_raw.to_dict()

    if not content_raw and not collab_raw:
        return f"No recommendations for '{movie_title}' and user {user_id}."

    # 3) Normalize (rank-based)
    content_norm = rank_norm(content_raw)
    collab_norm  = rank_norm(collab_raw)

    # 4) Candidate pool = union
    candidates = set(content_norm) | set(collab_norm)

    # 5) Blend
    final_scores = {
        mid: alpha*content_norm.get(mid,0) + (1-alpha)*collab_norm.get(mid,0)
        for mid in candidates
    }

    # 6) Exclude already rated + seed movie
    excluded = {seed_mid}
    if exclude_already_rated:
        excluded |= set(ratings[ratings['userId']==user_id]['movieId'].tolist())

    # 7) Sort + pick top-k
    sorted_final = sorted(
        [(mid, sc) for mid, sc in final_scores.items() if mid not in excluded],
        key=lambda x: x[1],
        reverse=True
    )[:k]

    if not sorted_final:
        return "No recommendations found."

    mids = [m for m,_ in sorted_final]
    mids = list(dict.fromkeys(mids))  # remove duplicates
    valid_mids = [m for m in mids if m in df[movie_id_col].values]

    # 8) Build output DataFrame
    out = df.set_index(movie_id_col).loc[valid_mids].reset_index()
    out['content_score'] = out[movie_id_col].map(content_norm).fillna(0.0)
    out['collab_score']  = out[movie_id_col].map(collab_norm).fillna(0.0)
    out['final_score']   = out[movie_id_col].map(dict(sorted_final))

    # 9) Return clean table
    cols = [movie_id_col, title_col, 'genres', 'cast', 'director',
            'content_score', 'collab_score', 'final_score']
    cols = [c for c in cols if c in out.columns]
    return out[cols]


In [ ]:
# 🔹 Test hybrid recommender with known movie + user
test_result = hybrid_recommend_fixed(
    user_id=1,
    movie_title="Under Siege 2: Dark Territory (1995)	",
    k=8,
    alpha=0.6  # balance between content & collaborative
)

test_result


,movieId_x,title,genres,cast,director,content_score,collab_score,final_score
0,10,GoldenEye (1995),Action|Adventure|Thriller,"Pierce Brosnan, Sean Bean, Izabella Scorupco",Martin Campbell,0.992481,0.996711,0.994173
1,95,Broken Arrow (1996),Action|Adventure|Thriller,"Adam West, David G. Jackson, Walter Gregg",Peter Kuran,0.991071,0.960526,0.978853
2,135,Down Periscope (1996),Comedy,"Kelsey Grammer, Lauren Holly, Rob Schneider",David S. Ward,0.994831,0.946898,0.975658
3,179,Mad Love (1995),Drama|Romance,"Chris O'Donnell, Drew Barrymore, Matthew Lillard",Antonia Bird,0.997650,0.929041,0.970207
4,71,Fair Game (1995),Action,"William Baldwin, Cindy Crawford, Steven Berkoff",Andrew Sipes,0.966635,0.970395,0.968139
5,151,Rob Roy (1995),Action|Drama|Romance|War,"Tim Elliot, Ron Haddrick, Jane Harders",Geoff Collins,0.984023,0.941729,0.967105
6,161,Crimson Tide (1995),Drama|Thriller|War,"Denzel Washington, Gene Hackman, Matt Craven",Tony Scott,0.986372,0.937030,0.966635
7,118,If Lucy Fell (1996),Comedy|Romance,"Sarah Jessica Parker, Eric Schaeffer, Ben Stiller",Eric Schaeffer,0.973684,0.951598,0.964850


# Model Evaluation